# Phase S1: Cooling Theory Validation

**Goal**: Validate BatchNorm and Skip as "cooling mechanisms"

**Design**:
- 4 configs: None / LN / BN / Skip
- 2 D values via base_ch: 32, 96
- 100 epochs per run
- **Checkpoint support** - saves every 20 epochs
- **Periodic logging** - prints progress every 10 epochs

**Expected time**: ~50 min on T4 GPU

In [ ]:
# Setup
import os, sys, json, math, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

os.chdir('/home/node/ThermoRG-NN' if os.path.exists('/home/node/ThermoRG-NN') else '.')
sys.path.insert(0, os.getcwd())

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}", flush=True)
print(f"CWD: {os.getcwd()}", flush=True)

In [ ]:
# Config
CONFIGS = [
    ('None_NoSkip',  'none',       False),
    ('LN_NoSkip',    'layernorm',  False),
    ('BN_NoSkip',    'batchnorm',  False),
    ('None_Skip',    'none',       True),
]

D_VALUES = [32, 96]  # base_ch
SEEDS = [42]
EPOCHS = 100
BATCH_SIZE = 128
LR = 0.01
CHECKPOINT_EVERY = 20  # checkpoint every 20 epochs
LOG_EVERY = 10        # log every 10 epochs

OUT_DIR = Path('./phase_s1_results_v2')
OUT_DIR.mkdir(exist_ok=True)

print(f"Configs: {len(CONFIGS)}, D: {D_VALUES}, Epochs: {EPOCHS}", flush=True)
print(f"Checkpoint every {CHECKPOINT_EVERY} epochs, log every {LOG_EVERY} epochs", flush=True)

In [ ]:
# Import ThermoNet from src
from thermorg.simulations.networks.thermonet import ThermoNet
from thermorg.simulations.networks.thermonet import J_topoComputation
from thermorg.scripts.compute_jtopo import compute_D_eff, compute_J_topo

print("ThermoNet imported successfully", flush=True)

In [ ]:
# Data
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])
transform_val = T.Compose([
    T.ToTensor(),
    T.Normalize([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
])

train_ds = CIFAR10('./data', train=True, transform=transform_train, download=True)
val_ds   = CIFAR10('./data', train=False, transform=transform_val, download=False)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Data: train={len(train_ds)}, val={len(val_ds)}", flush=True)

In [ ]:
# Training function with checkpoint + periodic logging
def train_one_model(model, epochs, lr, seed, checkpoint_path=None):
    """Train a single model with checkpoint support."""
    
    # Check for existing checkpoint
    start_epoch = 0
    best_loss = float('inf')
    
    if checkpoint_path and os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        start_epoch = ckpt['epoch'] + 1
        best_loss = ckpt.get('best_loss', float('inf'))
        print(f"  Resuming from epoch {start_epoch}, best_loss={best_loss:.4f}", flush=True)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    
    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    
    # Restore scheduler state
    for _ in range(start_epoch):
        scheduler.step()
    
    t0 = time.time()
    
    for epoch in range(start_epoch, epochs):
        # Train
        model.train()
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
        scheduler.step()
        
        # Evaluate
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for X, y in val_dl:
                X, y = X.to(device), y.to(device)
                val_loss += criterion(model(X), y).item() * X.size(0)
        val_loss /= len(val_ds)
        
        if val_loss < best_loss:
            best_loss = val_loss
        
        # Periodic logging
        if (epoch + 1) % LOG_EVERY == 0 or epoch == epochs - 1:
            elapsed = (time.time() - t0) / 60
            total_est = elapsed / (epoch - start_epoch + 1) * (epochs - start_epoch)
            remaining = total_est - elapsed
            print(f"  Epoch {epoch+1}/{epochs}: loss={val_loss:.4f}, elapsed={elapsed:.1f}min, remaining~{remaining:.1f}min", flush=True)
        
        # Checkpoint
        if checkpoint_path and (epoch + 1) % CHECKPOINT_EVERY == 0:
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'best_loss': best_loss,
            }, checkpoint_path)
            print(f"  Checkpoint saved at epoch {epoch+1}", flush=True)
    
    return best_loss

print("Training function defined", flush=True)

In [ ]:
# Main experiment loop
from datetime import datetime

results = {
    'timestamp': datetime.now().isoformat(),
    'epochs': EPOCHS,
    'configs': []
}

t_start = time.time()

for cfg_name, norm_type, use_skip in CONFIGS:
    print(f"\n{'='*60}", flush=True)
    print(f"[{cfg_name}] norm={norm_type}, skip={use_skip}", flush=True)
    print(f"{'='*60}", flush=True)
    
    cfg_start = time.time()
    cfg_result = {'name': cfg_name, 'norm': norm_type, 'skip': use_skip}
    
    for base_ch in D_VALUES:
        for seed in SEEDS:
            run_name = f"{cfg_name}_ch{base_ch}_s{seed}"
            ckpt_path = OUT_DIR / f"{run_name}.pt"
            
            print(f"\n--- {run_name} ---", flush=True)
            
            # Create model
            model = ThermoNet(
                in_channels=3,
                num_classes=10,
                base_channels=base_ch,
                depth=3,
                use_layernorm=(norm_type == 'layernorm'),
                use_batchnorm=(norm_type == 'batchnorm'),
                use_skip=use_skip,
                activation='gelu'
            )
            
            # Compute J_topo at init
            j_computer = J_topoComputation(model)
            J_topo_init = j_computer.compute_jtopo()
            print(f"  J_topo(init) = {J_topo_init:.4f}", flush=True)
            
            # Train with checkpoint
            t0 = time.time()
            best_loss = train_one_model(model, EPOCHS, LR, seed, ckpt_path)
            elapsed = (time.time() - t0) / 60
            
            print(f"  Finished! best_loss={best_loss:.4f}, time={elapsed:.1f}min", flush=True)
            
            # Store result
            if f'ch{base_ch}' not in cfg_result:
                cfg_result[f'ch{base_ch}'] = {}
            cfg_result[f'ch{base_ch}'][f's{seed}'] = {
                'best_val_loss': best_loss,
                'time_min': elapsed
            }
            cfg_result.setdefault('J_topo_init', J_topo_init)
            
            # Clear memory
            del model
            torch.cuda.empty_cache()
    
    results['configs'].append(cfg_result)
    cfg_time = (time.time() - cfg_start) / 60
    print(f"\n[{cfg_name}] total time: {cfg_time:.1f} min", flush=True)

total_time = (time.time() - t_start) / 60
results['total_time_min'] = total_time
print(f"\n{'='*60}", flush=True)
print(f"ALL DONE! Total time: {total_time:.1f} min", flush=True)
print(f"{'='*60}", flush=True)

In [ ]:
# Save results
out_file = OUT_DIR / 'phase_s1_results.json'
with open(out_file, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Saved to {out_file}", flush=True)

In [ ]:
# Summary
print("\n" + "="*60, flush=True)
print("SUMMARY", flush=True)
print("="*60, flush=True)

for cfg in results['configs']:
    losses = []
    for ch in D_VALUES:
        for s in SEEDS:
            if f'ch{ch}' in cfg:
                losses.append(cfg[f'ch{ch}'][f's{s}']['best_val_loss'])
    avg = sum(losses)/len(losses) if losses else 0
    print(f"{cfg['name']:<15} J={cfg.get('J_topo_init', 0):.4f}  avg_loss={avg:.4f}", flush=True)

print("\nDone!", flush=True)